# 08 · Convergência do ensemble — de quantas réplicas você precisa?

| | |
|---|---|
| **Autor** | Patrick Allan dos Santos Faustino |
| **Projeto** | Revisão tutorial de dinâmica molecular — xilanase GH11 (PDB 1XNB) |
| **Versão** | v2026.11 — notebook autocontido |

**Pergunta.** Com o número de réplicas que rodei, a média do ensemble já é precisa o bastante, e quantas seriam necessárias para um erro-alvo?

Wan, Sinclair & Coveney (*Phil. Trans. R. Soc. A* **2021**, 379, 20200082): **não há meio
teórico** de fixar quantas réplicas um estudo precisa; o critério é operacional, com o erro caindo
aproximadamente como $N^{-1/2}$. Este notebook **não lê trajetórias**: consome o CSV único de outro
notebook da suíte (`*_dados.csv`, tabela `serie`) ou médias digitadas.

**Painel A** — erro padrão × $N$ por subamostragem dos subconjuntos de tamanho $N$, com a lei
$N^{-1/2}$ ancorada em $N = K$. **Painel B** — média do ensemble × $N$.

> **A unidade estatística é a réplica, não o frame**: cada réplica contribui com **um** número, a
> sua média de produção.
>
> **Mede precisão, não exatidão.** Erro pequeno com réplicas presas na mesma bacia continua sendo
> erro pequeno em torno do valor errado.
>
> **Por que não há "critério do N + 1".** Subconjuntos sorteados das mesmas $K$ réplicas não testam
> réplicas novas: a média de subconjunto é não enviesada para qualquer $N$ e, para $N > K/2$, a faixa
> encolhe por construção (fator de população finita) — região sombreada nos gráficos. Em seu lugar,
> um dimensionamento amostral: o menor $N$ com $t_{N-1}\,s/\sqrt{N} \le$ `ERRO_ALVO_RELATIVO`·$|\bar x|$.

**Blocos:** 1 bibliotecas · 2 configuração e dados · 3 cálculos · 4 estatística ·
5 gráfico (matplotlib) · 6 gráfico interativo (Plotly) · 7 exportação.

## 1 · Bibliotecas

In [ ]:
import sys
import platform
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from itertools import combinations
from math import comb
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

INICIO = datetime.now()
VERSOES = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Plotly": plotly.__version__,
}
print(f"Data e hora da execução: {INICIO:%d/%m/%Y %H:%M:%S}")
print(f"Sistema operacional: {platform.platform()}")
df_versoes = pd.DataFrame(list(VERSOES.items()), columns=["biblioteca", "versao"])
df_versoes

## 2 · Configuração da análise e carregamento dos dados

In [ ]:
SISTEMA = "xilanase_1XNB"
# ── Entrada: o CSV único de outro notebook (ou MEDIAS_MANUAIS) ──────────────
ARQUIVO_DADOS = Path("resultados") / SISTEMA / "03_rmsd" / "rmsd_dados.csv"
TABELA = "serie"                     # linhas com tabela == "serie"
OBSERVAVEL = "RMSD Cα"               # obrigatório quando o CSV tem vários observáveis
SO_PRODUCAO = True                   # usa só fase == "producao"
MEDIAS_MANUAIS = None                # ex.: {"rep1": 0.152, "rep2": 0.147, ...} (tem precedência)
ROTULO_GRANDEZA = "RMSD Cα (nm)"

# ── Análise ─────────────────────────────────────────────────────────────────
MAX_SUBCONJUNTOS = 500               # acima disto, sorteia esse número de subconjuntos
SEMENTE = 20260905
NIVEL_CONFIANCA = 0.95
N_REAMOSTRAS_BOOTSTRAP = 10_000
ERRO_ALVO_RELATIVO = 0.01            # meia-largura do IC relativa à média (0,01 = ±1 %)

DIR_SAIDA = Path("resultados") / SISTEMA / "08_convergencia_replicas"
PREFIXO = "convergencia"
TITULO = "08 · Convergência do ensemble"

In [ ]:
if MEDIAS_MANUAIS:
    df_medias = pd.DataFrame({"replica": list(MEDIAS_MANUAIS), "media": list(MEDIAS_MANUAIS.values()), "n_pontos": np.nan})
    origem = "MEDIAS_MANUAIS"
else:
    if not ARQUIVO_DADOS.exists():
        raise FileNotFoundError(f"{ARQUIVO_DADOS} não existe — rode antes o notebook que o exporta.")
    dados = pd.read_csv(ARQUIVO_DADOS, comment="#", low_memory=False)
    dados = dados[dados["tabela"] == TABELA]
    presentes = list(dict.fromkeys(dados["observavel"]))
    if OBSERVAVEL is None and len(presentes) > 1:
        raise ValueError(f"O CSV tem {len(presentes)} observáveis ({presentes}): escolha um em OBSERVAVEL.")
    if OBSERVAVEL is not None:
        if OBSERVAVEL not in presentes:
            raise ValueError(f"OBSERVAVEL = '{OBSERVAVEL}' não está no CSV: {presentes}")
        dados = dados[dados["observavel"] == OBSERVAVEL]
    if SO_PRODUCAO and "fase" in dados:
        dados = dados[dados["fase"] == "producao"]
    g = dados.groupby("replica", sort=False)["valor"]
    df_medias = pd.DataFrame({"replica": g.mean().index, "media": g.mean().to_numpy(), "n_pontos": g.size().to_numpy()})
    origem = str(ARQUIVO_DADOS)
    print(f"Lido: {ARQUIVO_DADOS} | tabela '{TABELA}' | observável '{OBSERVAVEL}' | {len(dados)} pontos")
medias = df_medias["media"].to_numpy(dtype=float)
K = medias.size
if K < 3:
    raise ValueError(f"{K} réplica(s): a curva de convergência precisa de pelo menos 3.")
REPLICAS = list(df_medias["replica"])
df_medias

## 3 · Cálculos

Para cada $N$ de 2 a $K$: todos os subconjuntos de tamanho $N$ (ou `MAX_SUBCONJUNTOS` sorteados),
a média e o SEM de cada um; relata-se a mediana entre subconjuntos e a faixa 10–90 %.
Bootstrap percentil sobre as $K$ médias (reamostragem com reposição).

In [ ]:
sorteio = np.random.default_rng(SEMENTE)
t_crit = lambda n: stats.t.ppf(0.5 + NIVEL_CONFIANCA / 2, df=n - 1)


def subconjuntos(n):
    if comb(K, n) <= MAX_SUBCONJUNTOS:
        return [np.array(c) for c in combinations(range(K), n)]
    return [sorteio.choice(K, size=n, replace=False) for _ in range(MAX_SUBCONJUNTOS)]


linhas = []
for n in range(2, K + 1):
    grupos = subconjuntos(n)
    m_sub = np.array([medias[g].mean() for g in grupos])
    s_sub = np.array([medias[g].std(ddof=1) / np.sqrt(n) for g in grupos])
    linhas.append({"n_replicas": n, "n_subconjuntos": len(grupos), "exaustivo": comb(K, n) <= MAX_SUBCONJUNTOS,
                   "media_mediana": np.median(m_sub), "media_p10": np.percentile(m_sub, 10), "media_p90": np.percentile(m_sub, 90),
                   "sem_mediano": np.median(s_sub), "sem_p10": np.percentile(s_sub, 10), "sem_p90": np.percentile(s_sub, 90),
                   "ic_mediano": np.median(t_crit(n) * s_sub), "limitado_pelo_conjunto": n > K / 2})
df_curva = pd.DataFrame(linhas)
boot = sorteio.choice(medias, size=(N_REAMOSTRAS_BOOTSTRAP, K), replace=True).mean(axis=1)
df_curva

## 4 · Estatística

Média ± SEM entre réplicas (IC t de Student) e IC por bootstrap. Com $K < 10$, o bootstrap
percentil sai **estreito demais** (ignora a incerteza de $s$): relate o IC de Student.
Dimensionamento: o menor $N$ que leva a meia-largura do IC a `ERRO_ALVO_RELATIVO` da média, com o
$s$ observado (estimativa: $s$ de poucas réplicas tende a subestimar $\sigma$).

In [ ]:
def formatar(media, erro):
    """'média ± erro' com 2 algarismos significativos no erro (GUM, JCGM 100:2008 §7.2.6)."""
    if not (np.isfinite(erro) and erro > 0):
        return f"{media:.4g}"
    casas = max(0, 1 - int(np.floor(np.log10(abs(erro)))))
    return f"{media:.{casas}f} ± {erro:.{casas}f}"


_SOBRESCRITO = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")


def unidade_mpl(unidade):
    """Unidade para o matplotlib: '⁻³' vira mathtext ($^{-3}$). O sinal sobrescrito '⁻' (U+207B)
    falta em várias fontes (a Aptos Serif, inclusive) e sai como um quadrado na figura."""
    return re.sub(r"⁻([⁰¹²³⁴⁵⁶⁷⁸⁹]+)",
                  lambda m: "$^{-" + m.group(1).translate(_SOBRESCRITO) + "}$", unidade or "")


def rotulo_media(ref):
    """Rótulo dos painéis: 'média ± EP (N réplicas): valor ± erro unidade'.

    EP = erro padrão da média: entre as médias das réplicas (a réplica é a unidade estatística);
    com uma réplica só, o SEM da série corrigido pela ineficiência estatística."""
    n = ref.get("n", np.nan)
    origem = (f"{int(n)} réplicas" if ref.get("replica") == "entre_replicas" and pd.notna(n)
              else "1 réplica")
    return (f"média ± EP ({origem}): {formatar(ref['media'], ref['sem'])} "
            f"{unidade_mpl(ref.get('unidade', ''))}").strip()


s = float(medias.std(ddof=1))
sem = s / np.sqrt(K)
alfa = (1 - NIVEL_CONFIANCA) / 2
alvo = ERRO_ALVO_RELATIVO * abs(medias.mean())
n_necessario = next((n for n in range(2, 10_001) if t_crit(n) * s / np.sqrt(n) <= alvo), None)
df_stats = pd.DataFrame([{
    "observavel": OBSERVAVEL or ROTULO_GRANDEZA, "replica": "entre_replicas", "n": K, "media": medias.mean(),
    "desvio_padrao": s, "sem": sem, "ic_inferior": medias.mean() - t_crit(K) * sem,
    "ic_superior": medias.mean() + t_crit(K) * sem,
    "ic_bootstrap_inferior": np.percentile(boot, 100 * alfa), "ic_bootstrap_superior": np.percentile(boot, 100 * (1 - alfa)),
    "coef_variacao_pct": 100 * s / abs(medias.mean()), "assimetria": stats.skew(medias),
    "erro_alvo_relativo": ERRO_ALVO_RELATIVO, "n_replicas_para_erro_alvo": n_necessario}])
print(f"Média do ensemble: {formatar(medias.mean(), sem)} (IC {NIVEL_CONFIANCA:.0%} t: "
      f"[{df_stats['ic_inferior'][0]:.5g}, {df_stats['ic_superior'][0]:.5g}])")
if n_necessario is None:
    print(f"Nem 10 000 réplicas levariam o IC a ±{100 * ERRO_ALVO_RELATIVO:g} % com a dispersão atual.")
else:
    print(f"Para IC de ±{100 * ERRO_ALVO_RELATIVO:g} %: ~{n_necessario} réplica(s) (você tem {K}).")
if K < 10:
    print(f"[bootstrap] Com K = {K} o IC por bootstrap sai estreito demais: relate o de Student.")
df_stats.T.rename(columns={0: "valor"})

## 5 · Gráfico (matplotlib)

In [ ]:
# ── Estilo geral (vale para todas as figuras deste notebook) ────────────────
FONTE = "Arial"                 # se não estiver instalada, usa DejaVu Serif
TAMANHO_FONTE = 10                     # pt
CORES = ["#003f5c", "#bc5090", "#ffa600", "#58508d", "#ff6361"]   # uma por réplica
COR_MEDIA = "#333333"                 # tracejado da média
COR_REFERENCIA = "#d62728"            # linhas de referência e faixa do descarte

if FONTE not in {f.name for f in font_manager.fontManager.ttflist}:
    FONTE = "DejaVu Serif"
plt.rcParams.update({
    "font.family": FONTE, "font.size": TAMANHO_FONTE,
    "axes.labelsize": TAMANHO_FONTE, "axes.titlesize": TAMANHO_FONTE,
    "xtick.labelsize": TAMANHO_FONTE - 1, "ytick.labelsize": TAMANHO_FONTE - 1,
    "legend.fontsize": TAMANHO_FONTE - 1, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 120,
    "pdf.fonttype": 42,                # fontes embutidas como TrueType (editáveis)
    "svg.fonttype": "none",            # texto do SVG continua texto
})

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA = 6.3, 3.5
COR_ERRO, COR_MEDIA_N = "#003f5c", "#bc5090"
COR_IC = "#ffa600"                    # IC t da média das K réplicas (painel B)
COR_LIMITADO = "#969696"              # região N > K/2

# ── Gráfico ─────────────────────────────────────────────────────────────────
n = df_curva["n_replicas"].to_numpy()
ancora = float(df_curva["sem_mediano"].iloc[-1])
fig, (ax_e, ax_m) = plt.subplots(1, 2, figsize=(LARGURA, ALTURA), layout="constrained")
ax_e.fill_between(n, df_curva["sem_p10"], df_curva["sem_p90"], color=COR_ERRO, alpha=0.2, lw=0,
                  label="EP: 10–90 % entre subconjuntos")
ax_e.plot(n, df_curva["sem_mediano"], color=COR_ERRO, marker="o", ms=3.5, lw=1.0, label="EP mediano")
ax_e.plot(n, ancora * np.sqrt(K / n), color=COR_MEDIA, ls="--", lw=0.8, label="lei $N^{-1/2}$")
ax_e.set(xlabel="Número de réplicas $N$", ylabel=f"Erro padrão (EP) — {ROTULO_GRANDEZA}", xscale="log", yscale="log")
ax_m.fill_between(n, df_curva["media_p10"], df_curva["media_p90"], color=COR_MEDIA_N, alpha=0.2, lw=0,
                  label="média: 10–90 % entre subconjuntos")
ax_m.plot(n, df_curva["media_mediana"], color=COR_MEDIA_N, marker="o", ms=3.5, lw=1.0, label="média: mediana")
ax_m.axhline(medias.mean(), color=COR_MEDIA, ls="--", lw=0.7,
             label=f"{K} réplicas: {formatar(medias.mean(), sem)} (média ± EP)")
ax_m.axhspan(df_stats["ic_inferior"][0], df_stats["ic_superior"][0], color=COR_IC, alpha=0.15, lw=0,
             label=f"IC {NIVEL_CONFIANCA:.0%} (t, {K - 1} g.l.) da média")
ax_m.set(xlabel="Número de réplicas $N$", ylabel=ROTULO_GRANDEZA)
for eixo, letra in zip((ax_e, ax_m), "AB"):
    eixo.axvspan(K / 2, K + 0.5, color=COR_LIMITADO, alpha=0.12, lw=0,
                 label=f"N > K/2: faixa estreita por construção")
    eixo.set(xticks=n, xlim=(1.5, K + 0.5))
    eixo.xaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
    eixo.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    eixo.legend(frameon=True, fontsize=TAMANHO_FONTE - 2)
    eixo.set_title(letra, loc="left", fontweight="bold")
fig_convergencia = fig
plt.show()

## 6 · Gráfico interativo (Plotly)

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_PX = 430                # pixels
LARGURA_PX = None              # None = largura da página
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
fig_i = make_subplots(rows=1, cols=2, horizontal_spacing=0.12, subplot_titles=("A — erro padrão × N", "B — média × N"))
for col, (p0, p1, centro, cor, nome) in enumerate((("sem_p10", "sem_p90", "sem_mediano", CORES[0], "SEM mediano"),
                                                    ("media_p10", "media_p90", "media_mediana", CORES[1], "média (mediana)")), start=1):
    fig_i.add_trace(go.Scatter(x=np.r_[n, n[::-1]], y=np.r_[df_curva[p1], df_curva[p0][::-1]], fill="toself", mode="lines",
                               fillcolor="rgba(150,150,150,0.25)", line=dict(width=0), hoverinfo="skip",
                               name="10–90 %", showlegend=(col == 1)), row=1, col=col)
    fig_i.add_trace(go.Scatter(x=n, y=df_curva[centro], mode="lines+markers", name=nome, line=dict(color=cor)), row=1, col=col)
    fig_i.add_vrect(x0=K / 2, x1=K + 0.5, fillcolor="#969696", opacity=0.12, line_width=0, row=1, col=col,
                    annotation_text="N > K/2", annotation_position="top left")
fig_i.add_trace(go.Scatter(x=n, y=ancora * np.sqrt(K / n), mode="lines", name="N<sup>-1/2</sup>",
                           line=dict(color=COR_MEDIA, dash="dash")), row=1, col=1)
fig_i.add_hline(y=medias.mean(), line=dict(color=COR_MEDIA, dash="dash"), row=1, col=2)
fig_i.update_xaxes(title_text="Número de réplicas N", tickvals=n, range=[1.5, K + 0.5])
fig_i.update_yaxes(title_text=f"Erro padrão — {ROTULO_GRANDEZA}", type="log", row=1, col=1)
fig_i.update_xaxes(type="log", range=[np.log10(1.5), np.log10(K + 0.5)], row=1, col=1)   # log-log: N^-1/2 é uma reta
fig_i.update_yaxes(title_text=ROTULO_GRANDEZA, row=1, col=2)
fig_i.update_layout(template=TEMPLATE, height=ALTURA_PX, width=LARGURA_PX, font=dict(family=FONTE, size=TAMANHO_FONTE_PX))
fig_interativo = fig_i
fig_i.show()

## 7 · Exportação

Tudo vai para `DIR_SAIDA`, separado por formato:

```
{DIR_SAIDA}/
├── png/   600 dpi
├── pdf/   vetorial, fontes embutidas
├── svg/   vetorial, texto editável
├── html/  gráficos interativos (Plotly)
└── <prefixo>_dados.csv   ← CSV ÚNICO com todos os dados
```

O CSV é **um só**, em formato longo: a coluna `tabela` diz de que tabela vem
cada linha; as colunas que não se aplicam ficam vazias. As linhas iniciadas por
`#` registram data, hora, versões e parâmetros. Para ler:
`pd.read_csv(arquivo, comment="#")`.

| `tabela` | conteúdo |
|---|---|
| `medias` | média de produção de cada réplica (a entrada da análise) |
| `curva` | por N: média e SEM medianos entre subconjuntos e faixas 10–90 % |
| `estatistica` | média ± SEM, IC t e bootstrap, coeficiente de variação, réplicas para o erro-alvo |

In [ ]:
FORMATOS = ("png", "pdf", "svg")
DPI = 600
FUNDO_TRANSPARENTE = False
PLOTLY_JS = "cdn"                  # "cdn" = HTML leve; True = embute o plotly.js (abre offline)
EXPORTAR_PLOTLY_ESTATICO = False   # True = também PNG/PDF/SVG do Plotly (precisa do kaleido)

FIGURAS = {"convergencia_replicas": fig_convergencia}
INTERATIVOS = {"convergencia_replicas": fig_interativo}
TABELAS = {"medias": df_medias, "curva": df_curva, "estatistica": df_stats}
PARAMETROS = {"origem": origem, "observavel": OBSERVAVEL, "max_subconjuntos": MAX_SUBCONJUNTOS,
              "n_bootstrap": N_REAMOSTRAS_BOOTSTRAP, "erro_alvo_relativo": ERRO_ALVO_RELATIVO}

# ── Figuras ─────────────────────────────────────────────────────────────────
for nome, fig in FIGURAS.items():
    for ext in FORMATOS:
        (DIR_SAIDA / ext).mkdir(parents=True, exist_ok=True)
        fig.savefig(DIR_SAIDA / ext / f"{nome}.{ext}", dpi=DPI, bbox_inches="tight",
                    transparent=FUNDO_TRANSPARENTE)
for nome, fig in INTERATIVOS.items():
    (DIR_SAIDA / "html").mkdir(parents=True, exist_ok=True)
    fig.write_html(DIR_SAIDA / "html" / f"{nome}.html", include_plotlyjs=PLOTLY_JS,
                   config={"displaylogo": False, "toImageButtonOptions": {"format": "svg"}})
    if EXPORTAR_PLOTLY_ESTATICO:
        for ext in FORMATOS:
            try:
                fig.write_image(DIR_SAIDA / ext / f"{nome}_interativo.{ext}", scale=3)
            except Exception as erro:
                print(f"[aviso] {nome}.{ext} do Plotly não exportado: {erro}")
                break

# ── CSV único ───────────────────────────────────────────────────────────────
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
partes = []
for nome, tabela in TABELAS.items():
    t = tabela.copy()
    t.insert(0, "tabela", nome)
    partes.append(t)
dados = pd.concat(partes, ignore_index=True, sort=False)
arquivo_csv = DIR_SAIDA / f"{PREFIXO}_dados.csv"
with open(arquivo_csv, "w", encoding="utf-8", newline="") as f:
    f.write(f"# {TITULO}\n")
    f.write(f"# gerado em {INICIO:%Y-%m-%d %H:%M:%S} | sistema {SISTEMA} | réplicas {REPLICAS}\n")
    f.write("# versoes: " + "; ".join(f"{k} {v}" for k, v in VERSOES.items()) + "\n")
    f.write("# parametros: " + "; ".join(f"{k} = {v}" for k, v in PARAMETROS.items()) + "\n")
    f.write("# tabelas: " + ", ".join(f"{k} ({len(v)} linhas)" for k, v in TABELAS.items()) + "\n")
    dados.to_csv(f, index=False)

# ── O que foi gravado ───────────────────────────────────────────────────────
df_exportados = pd.DataFrame([{"pasta": str(p.parent.relative_to(DIR_SAIDA)), "arquivo": p.name,
                               "tamanho_kB": round(p.stat().st_size / 1024, 1)}
                              for p in sorted(DIR_SAIDA.rglob("*")) if p.is_file()])
print(f"Resultados em: {DIR_SAIDA.resolve()}")
print(f"Duração da execução: {datetime.now() - INICIO}")
df_exportados